# CSP Experiments for Motor Imagery Classification

This notebook investigates Common Spatial Patterns (CSP) for classifying left-hand (T1) and right-hand (T2) motor imagery.

CSP + LDA is evaluated using Leave-One-Subject-Out cross-validation, first with three motor-related EEG channels and then with all 64 channels.

The goal is to determine whether spatial EEG features improve cross-subject classification compared with handcrafted features.

## 1. EEG Epoch Preparation

Load recordings from five subjects and extract 4-second motor imagery epochs after average referencing and 8–30 Hz filtering.

Unlike handcrafted features, CSP operates directly on multichannel EEG epochs.

In [1]:
from pathlib import Path
import mne
import numpy as np
import pandas as pd

subjects = [f"S{i:03d}" for i in range(1, 6)]
runs = ["04", "08", "12"]
motor_channels = ["C3..", "Cz..", "C4.."]
data_dir = Path("../data/raw")

X_list = []
y_list = []
groups_list = []

for subject in subjects:
    for run in runs:
        recording_id = f"{subject}R{run}"
        raw_path = data_dir / f"{recording_id}.edf"

        raw = mne.io.read_raw_edf(raw_path, preload=True, verbose=False)
        raw.set_eeg_reference(ref_channels="average", projection=False, verbose=False)
        raw.filter(l_freq=8.0, h_freq=30.0, picks="eeg", verbose=False)

        events, event_id = mne.events_from_annotations(raw, verbose=False)

        epochs = mne.Epochs(raw, events, event_id={"T1": event_id["T1"], "T2": event_id["T2"]},
                            tmin=0.0, tmax=4.0, picks=motor_channels,
                            baseline=None, preload=True, reject_by_annotation=True, verbose=False)

        X_list.append(epochs.get_data())
        y_list.append(np.where(epochs.events[:, -1] == event_id["T1"], 0, 1))
        groups_list.extend([subject] * len(epochs))

        print(f"{recording_id}: {len(epochs)} epochs")

X_epochs = np.concatenate(X_list, axis=0)
y_csp = np.concatenate(y_list)
groups_csp = np.array(groups_list)

print("\nEEG epochs shape:", X_epochs.shape)
print("Class distribution:", np.bincount(y_csp))
print("Subjects:", np.unique(groups_csp))

S001R04: 15 epochs
S001R08: 15 epochs
S001R12: 15 epochs
S002R04: 15 epochs
S002R08: 15 epochs
S002R12: 15 epochs
S003R04: 15 epochs
S003R08: 15 epochs
S003R12: 15 epochs
S004R04: 15 epochs
S004R08: 15 epochs
S004R12: 15 epochs
S005R04: 15 epochs
S005R08: 15 epochs
S005R12: 15 epochs

EEG epochs shape: (225, 3, 641)
Class distribution: [113 112]
Subjects: ['S001' 'S002' 'S003' 'S004' 'S005']


## 2. CSP + LDA: Three EEG Channels

Evaluate a CSP + LDA pipeline using channels C3, Cz, and C4.

CSP extracts two spatial features from each EEG epoch. Leave-One-Subject-Out cross-validation is used to assess cross-subject generalization, with CSP fitted only on the training data in each fold.
Performance is evaluated using accuracy, balanced accuracy, binary F1, and Macro-F1 to assess classification quality across both motor imagery classes.

In [2]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.pipeline import make_pipeline

logo = LeaveOneGroupOut()

csp_lda = make_pipeline(CSP(n_components=2, reg=None, log=True, norm_trace=False), LinearDiscriminantAnalysis())

scoring = {"accuracy": "accuracy",
           "balanced_accuracy": "balanced_accuracy",
           "f1": "f1",
           "f1_macro": "f1_macro"}

csp_cv_results = cross_validate(csp_lda, X_epochs, y_csp, groups=groups_csp, cv=logo, scoring=scoring)

csp_results = pd.DataFrame({"Metric": list(scoring),
                            "Mean": [csp_cv_results[f"test_{metric}"].mean() for metric in scoring],
                            "Std": [csp_cv_results[f"test_{metric}"].std() for metric in scoring]})

display(csp_results.round(3))

Computing rank from data with rank=None
    Using tolerance 2e-06 (2.2e-16 eps * 3 dim * 3e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 2e-06 (2.2e-16 eps * 3 dim * 3.1e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 1.9e-06 (2.2e-16 eps * 3 dim * 2.9e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating 

,Metric,Mean,Std
0,accuracy,0.564,0.057
1,balanced_accuracy,0.560,0.062
2,f1,0.528,0.188
3,f1_macro,0.493,0.091


### CSP Results: Three Channels

CSP + LDA with two components achieved a balanced accuracy of **0.560** and Macro-F1 of **0.493**.

Although balanced accuracy was comparable to handcrafted-feature LDA (0.558), Macro-F1 was lower (0.493 vs. 0.538).

Thus, two-component CSP did not demonstrate a consistent improvement over handcrafted features.

## 3. CSP with All EEG Channels

The three-channel CSP model showed no meaningful improvement over the handcrafted-feature baseline.

We now prepare epochs containing all 64 EEG channels to evaluate whether a broader spatial representation improves classification. The effect of average referencing on data rank will also be considered.

In [3]:
X_all_list = []

for subject in subjects:
    for run in runs:
        recording_id = f"{subject}R{run}"
        raw_path = data_dir / f"{recording_id}.edf"

        raw = mne.io.read_raw_edf(raw_path, preload=True, verbose=False)
        raw.set_eeg_reference(ref_channels="average", projection=False, verbose=False)
        raw.filter(l_freq=8.0, h_freq=30.0, picks="eeg", verbose=False)

        events, event_id = mne.events_from_annotations(raw, verbose=False)

        epochs = mne.Epochs(raw, events, event_id={"T1": event_id["T1"], "T2": event_id["T2"]}, tmin=0.0, tmax=4.0, picks="eeg",
                            baseline=None, preload=True, reject_by_annotation=True, verbose=False)

        X_all_list.append(epochs.get_data())
        print(f"{recording_id}: {len(epochs)} epochs, {len(epochs.ch_names)} channels")

X_epochs_all = np.concatenate(X_all_list, axis=0)

print("\nAll-channel EEG shape:", X_epochs_all.shape)
print("Labels:", y_csp.shape)
print("Groups:", groups_csp.shape)

S001R04: 15 epochs, 64 channels
S001R08: 15 epochs, 64 channels
S001R12: 15 epochs, 64 channels
S002R04: 15 epochs, 64 channels
S002R08: 15 epochs, 64 channels
S002R12: 15 epochs, 64 channels
S003R04: 15 epochs, 64 channels
S003R08: 15 epochs, 64 channels
S003R12: 15 epochs, 64 channels
S004R04: 15 epochs, 64 channels
S004R08: 15 epochs, 64 channels
S004R12: 15 epochs, 64 channels
S005R04: 15 epochs, 64 channels
S005R08: 15 epochs, 64 channels
S005R12: 15 epochs, 64 channels

All-channel EEG shape: (225, 64, 641)
Labels: (225,)
Groups: (225,)


### 4. CSP + LDA: All 64 Channels

Evaluate CSP + LDA using all 64 EEG channels with two spatial components.

Leave-One-Subject-Out cross-validation is applied to assess whether additional spatial information improves classification compared with the three-channel CSP model.

The effective data rank is set to 63 to account for average referencing.

In [4]:
csp_lda_all = make_pipeline(CSP(n_components=2, reg=None, log=True, norm_trace=False, rank=None), LinearDiscriminantAnalysis())

csp_all_scores = cross_validate(csp_lda_all, X_epochs_all, y_csp, groups=groups_csp, cv=logo, scoring=scoring)

csp_all_results = pd.DataFrame({"Metric": list(scoring), 
                                "Mean": [csp_all_scores[f"test_{metric}"].mean() for metric in scoring], 
                                "Std": [csp_all_scores[f"test_{metric}"].std() for metric in scoring]})

display(csp_all_results.round(3))

Computing rank from data with rank=None
    Using tolerance 0.0002 (2.2e-16 eps * 64 dim * 1.4e+10  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 0.00023 (2.2e-16 eps * 64 dim * 1.6e+10  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 0.0

,Metric,Mean,Std
0,accuracy,0.458,0.067
1,balanced_accuracy,0.453,0.057
2,f1,0.465,0.250
3,f1_macro,0.337,0.023


### CSP Results: All 64 Channels

CSP + LDA using 64 EEG channels achieved a balanced accuracy of **0.453** and Macro-F1 of **0.337**, substantially below the three-channel CSP model.

Performance remained close to or below the DummyClassifier reference, suggesting poor cross-subject generalization.

Adding more EEG channels did not improve classification with two CSP components.

## 5. CSP Component Selection

The 64-channel CSP model performed worse than the three-channel model when using two components.

We now evaluate different numbers of CSP components to investigate whether retaining more spatial information improves cross-subject classification.

All configurations are evaluated using the same Leave-One-Subject-Out splits.

In [9]:
component_configs = {"3 channels": (X_epochs, [2, 3]), 
                     "64 channels": (X_epochs_all, [2, 4, 6, 8])}
component_results = []

for channel_set, (X_data, components) in component_configs.items():
    for n_components in components:
        model = make_pipeline(CSP(n_components=n_components, reg=None, log=True, norm_trace=False, rank=None), 
                              LinearDiscriminantAnalysis())

        cv_results = cross_validate(model, X_data, y_csp, groups=groups_csp, cv=logo, scoring=scoring)

        component_results.append({"Channels": channel_set,
                                  "Components": n_components,
                                  "Accuracy": cv_results["test_accuracy"].mean(),
                                  "Balanced Accuracy": cv_results["test_balanced_accuracy"].mean(),
                                  "F1": cv_results["test_f1"].mean(),
                                  "Macro-F1": cv_results["test_f1_macro"].mean(),
                                  "Balanced Accuracy Std": cv_results["test_balanced_accuracy"].std()})
component_df = pd.DataFrame(component_results)

display(component_df.round(3))

Computing rank from data with rank=None
    Using tolerance 2e-06 (2.2e-16 eps * 3 dim * 3e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 2e-06 (2.2e-16 eps * 3 dim * 3.1e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 1.9e-06 (2.2e-16 eps * 3 dim * 2.9e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating 

,Channels,Components,Accuracy,Balanced Accuracy,F1,Macro-F1,Balanced Accuracy Std
0,3 channels,2,0.564,0.560,0.528,0.493,0.062
1,3 channels,3,0.573,0.577,0.619,0.525,0.079
2,64 channels,2,0.458,0.453,0.465,0.337,0.057
3,64 channels,4,0.493,0.486,0.384,0.341,0.028
4,64 channels,6,0.489,0.477,0.303,0.383,0.078
5,64 channels,8,0.524,0.516,0.411,0.397,0.015


### CSP Component Selection Results

Three-channel CSP with three components achieved the highest balanced accuracy (**0.577**) and Macro-F1 (**0.525**) among the tested CSP configurations.

Increasing the number of components improved performance for the three-channel model, while all 64-channel configurations performed worse.

Compared with handcrafted-feature LDA, the best CSP model achieved slightly higher balanced accuracy (0.577 vs. 0.558) but lower Macro-F1 (0.525 vs. 0.538).

Neither approach demonstrated a consistent advantage across both metrics.

## 6. Regularized CSP

The best CSP configuration used three EEG channels and three spatial components, achieving a balanced accuracy of 0.577.

We now test covariance regularization to investigate whether more stable covariance estimates improve cross-subject classification.

All configurations use the same Leave-One-Subject-Out cross-validation strategy.

In [10]:
regularization_values = [None, 0.1, 0.5]
regularization_results = []

for reg in regularization_values:
    model = make_pipeline(CSP(n_components=3, reg=reg, log=True, norm_trace=False, rank=None),
                          LinearDiscriminantAnalysis())

    cv_results = cross_validate(model, X_epochs, y_csp, groups=groups_csp, cv=logo, scoring=scoring)

    regularization_results.append({"Regularization": str(reg),
                                   "Accuracy": cv_results["test_accuracy"].mean(),
                                   "Balanced Accuracy": cv_results["test_balanced_accuracy"].mean(),
                                   "F1": cv_results["test_f1"].mean(),
                                   "Macro-F1": cv_results["test_f1_macro"].mean(),
                                   "Balanced Accuracy Std": cv_results["test_balanced_accuracy"].std()})

regularization_df = pd.DataFrame(regularization_results)

display(regularization_df.round(3))

Computing rank from data with rank=None
    Using tolerance 2e-06 (2.2e-16 eps * 3 dim * 3e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 2e-06 (2.2e-16 eps * 3 dim * 3.1e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 1.9e-06 (2.2e-16 eps * 3 dim * 2.9e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating 

,Regularization,Accuracy,Balanced Accuracy,F1,Macro-F1,Balanced Accuracy Std
0,None,0.573,0.577,0.619,0.525,0.079
1,0.1,0.542,0.545,0.556,0.482,0.085
2,0.5,0.560,0.562,0.544,0.515,0.085


### Regularized CSP Results

Covariance regularization did not improve cross-subject classification.

The unregularized CSP model achieved the highest balanced accuracy (**0.577**) and Macro-F1 (**0.525**).

Both regularized configurations performed worse and showed slightly higher variability across subjects.

Therefore, the unregularized three-channel CSP model with three components remains the best exploratory CSP configuration.

## 7. Subject-Level CSP Evaluation

The unregularized three-channel CSP model achieved the best exploratory performance.

We now examine balanced accuracy for each held-out subject to identify individual differences in cross-subject generalization.

In [11]:
best_csp_model = make_pipeline(CSP(n_components=3, reg=None, log=True, norm_trace=False, rank=None),
                               LinearDiscriminantAnalysis())

subject_cv = cross_validate(best_csp_model, X_epochs, y_csp, groups=groups_csp, cv=logo, scoring=scoring)

subject_results = pd.DataFrame({"Subject": np.unique(groups_csp),
                                "Accuracy": subject_cv["test_accuracy"],
                                "Balanced Accuracy": subject_cv["test_balanced_accuracy"],
                                "F1": subject_cv["test_f1"],
                                "Macro-F1": subject_cv["test_f1_macro"]})

display(subject_results.round(3))

Computing rank from data with rank=None
    Using tolerance 2e-06 (2.2e-16 eps * 3 dim * 3e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 2e-06 (2.2e-16 eps * 3 dim * 3.1e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 1.9e-06 (2.2e-16 eps * 3 dim * 2.9e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating 

,Subject,Accuracy,Balanced Accuracy,F1,Macro-F1
0,S001,0.644,0.650,0.714,0.622
1,S002,0.667,0.661,0.545,0.641
2,S003,0.489,0.499,0.646,0.363
3,S004,0.600,0.609,0.710,0.533
4,S005,0.467,0.467,0.478,0.466


### Subject-Level CSP Results

CSP achieved balanced accuracy above 0.60 for S001, S002, and S004, while performance for S003 and S005 remained near or below chance level.

S003 showed a particularly large difference between binary F1 (0.646) and Macro-F1 (0.363), indicating uneven classification performance across classes.

These results highlight substantial subject-specific variability and limited cross-subject generalization.

### 8. CSP vs. Handcrafted Features by Subject

We compare the best exploratory CSP configuration with handcrafted-feature LDA using the same held-out subjects.

Balanced accuracy and Macro-F1 are evaluated to determine whether CSP provides consistent improvements across participants and classes.

In [12]:
from sklearn.preprocessing import StandardScaler

handcrafted_df = pd.read_csv("../data/processed/eeg_features_all_subjects.csv")
handcrafted_df = handcrafted_df[handcrafted_df["event"].isin(["T1", "T2"])].copy()

features_to_drop = ["C3.._rms", "Cz.._rms", "C4.._rms", "C3.._relative_beta_power", "Cz.._relative_beta_power", "C4.._relative_beta_power"]

X_handcrafted = handcrafted_df.drop(columns=["subject_id", "recording_id", "event"] + features_to_drop)

y_handcrafted = handcrafted_df["event"].map({"T1": 0, "T2": 1}).to_numpy()
groups_handcrafted = handcrafted_df["subject_id"].to_numpy()

assert np.array_equal(y_handcrafted, y_csp)
assert np.array_equal(groups_handcrafted, groups_csp)

handcrafted_model = make_pipeline(StandardScaler(),
                                  LinearDiscriminantAnalysis())

handcrafted_cv = cross_validate(handcrafted_model, X_handcrafted, y_handcrafted, groups=groups_handcrafted, cv=logo, scoring=scoring)

comparison_subject_df = subject_results[["Subject", "Balanced Accuracy", "Macro-F1"]
    ].rename(columns={"Balanced Accuracy": "CSP BA",
                      "Macro-F1": "CSP Macro-F1"})

comparison_subject_df["Handcrafted BA"] = handcrafted_cv["test_balanced_accuracy"]
comparison_subject_df["Handcrafted Macro-F1"] = handcrafted_cv["test_f1_macro"]

comparison_subject_df["BA Difference"] = (comparison_subject_df["CSP BA"] - comparison_subject_df["Handcrafted BA"])

display(comparison_subject_df.round(3))

,Subject,CSP BA,CSP Macro-F1,Handcrafted BA,Handcrafted Macro-F1,BA Difference
0,S001,0.650,0.622,0.488,0.488,0.162
1,S002,0.661,0.641,0.618,0.603,0.043
2,S003,0.499,0.363,0.516,0.491,-0.017
3,S004,0.609,0.533,0.652,0.600,-0.043
4,S005,0.467,0.466,0.518,0.509,-0.051


### CSP vs. Handcrafted Features Results

CSP outperformed handcrafted-feature LDA on both balanced accuracy and Macro-F1 for S001 and S002.

However, handcrafted LDA performed better for S003, S004, and S005.

The small average balanced accuracy advantage of CSP was largely driven by S001, indicating inconsistent improvements across subjects.

Neither approach demonstrated a clear, consistent advantage in cross-subject generalization.

### 9. EEG Epoch Consistency Check

We verify that the three-channel and 64-channel datasets contain identical signals for C3, Cz, and C4.

This ensures that both CSP experiments use consistent epoch ordering and preprocessing.

In [14]:
channel_names = mne.io.read_raw_edf(data_dir / "S001R04.edf", preload=False, verbose=False).ch_names

channel_indices = [channel_names.index(ch) for ch in motor_channels]

X_selected = X_epochs_all[:, channel_indices, :]

print("Three-channel shape:", X_epochs.shape)
print("Selected from 64 channels:", X_selected.shape)

max_difference = np.max(np.abs(X_epochs - X_selected))

print("Maximum difference (µV):", max_difference * 1e6)

assert X_epochs.shape == X_selected.shape
assert np.allclose(X_epochs, X_selected, rtol=1e-6, atol=1e-12)

print("Epoch consistency check passed.")

Three-channel shape: (225, 3, 641)
Selected from 64 channels: (225, 3, 641)
Maximum difference (µV): 0.0
Epoch consistency check passed.


### EEG Epoch Consistency Results

The selected C3, Cz, and C4 signals were identical in the three-channel and 64-channel datasets (maximum difference: 0.0 µV).

This confirms consistent epoch ordering and preprocessing across both CSP experiments.

## Conclusion

Common Spatial Patterns (CSP) were evaluated for cross-subject classification of left- and right-hand motor imagery using Leave-One-Subject-Out cross-validation.

The best exploratory CSP configuration used three EEG channels (C3, Cz, C4), three spatial components, and no covariance regularization.

**Model comparison:**

| Model | Balanced Accuracy | Macro-F1 |
|---|---:|---:|
| Handcrafted Features + LDA | 0.558 | 0.538 |
| CSP + LDA | 0.577 | 0.525 |

Increasing the number of EEG channels did not improve classification, and covariance regularization provided no additional benefit.

CSP outperformed handcrafted-feature LDA for two of five subjects, while handcrafted features performed better for the remaining three.

An epoch consistency check confirmed identical C3, Cz, and C4 signals across the three-channel and 64-channel datasets.

**Limitations:** Performance remained close to chance level for some subjects. Furthermore, CSP parameters were selected using exploratory cross-validation on the same five subjects, so the reported results should not be interpreted as independent final test performance.

**Next step:** Evaluate model generalization using a larger subject population and a validation strategy that separates hyperparameter selection from performance estimation.